# 01 - Connectome Data Exploration

This notebook queries and analyzes the *Drosophila melanogaster* connectome from HHMI Janelia's `male-cns:v1.0` dataset, focusing on the Central Complex (CX) navigation neuropils.


In [ ]:
import sys, os
from pathlib import Path

# Add project root and research/ to sys.path
cwd = Path.cwd().resolve()
for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / 'research' / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
    if (p / 'src').is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from src.config import CONFIG
    from src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from src.environment import FlyNavigationEnv
    from src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
except ModuleNotFoundError:
    from research.src.config import CONFIG
    from research.src.connectome_utils import ConnectomeData, ConnectomeAnalysis, prune_graph_by_weight_threshold
    from research.src.environment import FlyNavigationEnv
    from research.src.agent import ConnectomeConstrainedAgent, BaselineAgent, RandomWeightAgent, PPOTrainer
    from research.src.metrics import MetricsLogger, analyze_neuron_activations, plot_activation_distribution, compute_path_efficiency
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt


## 1. Connectome Ingestion & Graph Construction
We load neuron metadata and synaptic connectivity (filtering for connections with at least 5 synapses).


In [ ]:
data = ConnectomeData()
neurons_df = data.fetch_all_neurons()
conn_df = data.fetch_connectivity(min_synapses=CONFIG['synapse_threshold'])
graph = data.build_networkx_graph(conn_df)

print(f"Total Neurons in Dataset: {len(neurons_df):,}")
print(f"Total Synaptic Edges: {len(conn_df):,}")


## 2. Central Complex (CX) Subgraph Extraction


In [ ]:
analysis = ConnectomeAnalysis(graph, neurons_df)
cx_graph = analysis.get_subgraph_by_roi('CX')
stats = analysis.print_statistics()

for k, v in stats.items():
    print(f"{k}: {v}")


## 3. Degree Distribution & Centrality Analysis
Examining power-law in/out degree distribution and identifying top hub neurons.


In [ ]:
centrality_df = analysis.compute_centrality(metric='degree')
print("Top 10 Hub Neurons by Degree:")
print(centrality_df.head(10))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
in_degs = [d for _, d in cx_graph.in_degree()]
out_degs = [d for _, d in cx_graph.out_degree()]

ax1.hist(in_degs, bins=30, color='#5B8CFF', alpha=0.8, edgecolor='#2A2A32')
ax1.set_title('In-Degree Distribution')
ax1.set_xlabel('In-Degree')
ax1.set_ylabel('Neuron Count')

ax2.hist(out_degs, bins=30, color='#FFB84D', alpha=0.8, edgecolor='#2A2A32')
ax2.set_title('Out-Degree Distribution')
ax2.set_xlabel('Out-Degree')
ax2.set_ylabel('Neuron Count')

plt.tight_layout()
plt.show()
